# import

In [ ]:
from utils.recbole_train_test import *
from utils.plot_utils import *
from utils.model_utils import get_trainer
from utils.custom_trainer import *
from utils.generate_artificial_random_dataset import load_picklefile

# functions

In [ ]:
def get_full_models_results_matrix(model_name,
                       models_versions, 
                       base_dataset_name, 
                       save_path, 
                       metric, 
                       filename_version):
    
    cols = [i for i in range(len(models_versions))]
    df = pd.DataFrame(columns=cols) 

    
    for row_idx, model_ver in enumerate(models_versions):
        print('\n\n', row_idx, model_ver)
        # print('\n\n',model_ver)
        current_dataset_name = base_dataset_name+model_ver

        for column_idx, section in enumerate(models_versions):
            file_dir = save_path+current_dataset_name+\
                                '/'+get_evaluation_results_filename(model_name,
                                                                    current_dataset_name,
                                                                    section,
                                                                    filename_version)+'.pkl'
            print(file_dir, 'row_idx='+str(row_idx)+'; column_idx='+str(column_idx))
            test_recall = load_picklefile(file_dir)[metric]

            df.loc[row_idx,column_idx] = test_recall

    df = df.apply(pd.to_numeric, errors='coerce')  
    return df

In [ ]:
def recbole_train_each_eval_all(model_name,
                                model_versions,                              
                                save_path, base_filename, specs_str,
                                filename_version,

                                use_gpu,
                                seed,
                                show_progress,
                                save_dataset,
                                shuffle,
                                benchmark_filename,
                                eval_args,
                                metrics,
                                ks,
                                valid_metric,
                            
                                part_shift_incl):

    model_class = None
    if model_name=='BPR':
        model_class = BPR
    elif model_name=='Pop':
        model_class = Pop
    elif model_name=='NeuMF':
        model_class = NeuMF
    elif model_name=='ItemKNN':
        model_class = ItemKNN
    else:
        raise Exception('Model name not expected! Current options are [BPR, Pop, NeuMF, ItemKNN]')  


    base_dataset_name = base_filename+'_'+specs_str

    for part in model_versions:
        print('\n\n'+part)
        
        # current data (ith part of the dataset) to feed the pre-trained model,
        # the result is referred to as "model part i" or "current model"
        current_dataset_name = base_dataset_name+part
        current_checkpoint_dir = save_path+current_dataset_name

        parameter_dict = {  'dataset': current_dataset_name+'.inter',
                            'use_gpu':use_gpu,

                            ## Environment settings https://recbole.io/docs/user_guide/config/environment_settings.html
                            'seed':seed,
                            'state':'INFO' if show_progress else 'ERROR', # ['INFO', 'DEBUG', 'WARNING', 'ERROR', 'CRITICAL']
                            'data_path': save_path, # The path of input dataset.
                            'save_dataset':save_dataset, 
                            'checkpoint_dir':current_checkpoint_dir, # The path to save checkpoint file.
                            'show_progress': show_progress,
                            'shuffle': shuffle,

                            ## Data settings https://recbole.io/docs/user_guide/config/data_settings.html
                            'load_col': {'inter': ['user_id', 'item_id', 'timestamp']},
                            # 'user_inter_num_interval':'[1,inf)',
                            'benchmark_filename': benchmark_filename,
                            
                            ## Training settings https://recbole.io/docs/user_guide/config/training_settings.html
                            # 'train_neg_sample_args': TRAIN_NEG_SAMPLE_ARGS,
                            
                            ## Evaluation settings https://recbole.io/docs/user_guide/config/evaluation_settings.html
                            'eval_args': eval_args,
                            'metrics': metrics, 
                            'topk':ks,
                            'valid_metric':valid_metric          
                        }


        # current_config,current_logger,current_dataset,current_train_data, current_valid_data, current_test_data
        current_config,\
            current_logger, _,\
                current_train_data,\
                    current_valid_data,\
                        current_test_data = setup_config_and_dataset(model_name,
                                                                    current_dataset_name,
                                                                    parameter_dict)

        # model loading and initialization
        current_model = model_class(current_config, current_train_data.dataset).to(current_config['device'])
        current_logger.info(current_model)


        # trainer loading and initialization
        trainer = Trainer(current_config, current_model)
        
        
        # model training
        best_valid_score, best_valid_result = trainer.fit(current_train_data, current_valid_data)
        print('\n\nTraining best results')
        print('best_valid_score: ', best_valid_score)
        print('best_valid_result: ', best_valid_result)


        # main diagonal eval
        test_result = evaluate(trainer, current_test_data)
        save_evaluation_results(test_result, 
                                current_config['checkpoint_dir'], 
                                get_evaluation_results_filename(current_config['model'], 
                                                                current_dataset_name, 
                                                                part, 
                                                                filename_version))


        # evaluate in all testsets
        test_full_data_sections = get_test_full_data_sections_with_names(model_version=part,
                                                base_dataset_name=base_dataset_name,
                                                models_versions=model_versions)
        
        test_full_data_sections = test_full_data_sections if part_shift_incl else test_full_data_sections[:-1]
        print(test_full_data_sections)

        for testset_name in test_full_data_sections:

            _,_,\
                _,trainset,_,\
                    testset = setup_config_and_dataset(model_name,
                                                        testset_name,
                                                        parameter_dict)

            # When calculate ItemCoverage metrics, we need to run this code for set item_nums in eval_collector.
            trainer.eval_collector.data_collect(trainset)
            
            # model evaluation
            test_result = evaluate(trainer, testset) # bc is the trainer that was just feed new data
            save_evaluation_results(test_result, 
                                    current_config['checkpoint_dir'],
                                    get_evaluation_results_filename(current_config['model'], 
                                                                    current_dataset_name, 
                                                                    testset_name, 
                                                                    filename_version))

# Experiment Goodreads - ET RD LS.t UD SF TO UM.100
executed train, random drift, leave one out train sampled, uniform neg sample training distribution, shuffle false, time ordering, uniform mode


In [ ]:
save_path, base_filename, specs_str = ('processed_datasets/natural_data/goodreads/inter_dedup_coldstart_3stars_4x714k/',
                                        'more_2interQ_df', 
                                        'NPT_RD.50')
BENCHMARK_FILENAMES = ['train', 'valid', 'test']
base_dataset_name = base_filename+'_'+specs_str

# variables

In [ ]:
# MODEL_VERSIONS = ['_pt1', '_pt2', '_pt3', '_pt4']
freq=6 # month
duration = 2*12//freq # 2 years split in xM buckets
n_parts = duration*2+1
d_keys = ['_pt'+str(i) for i in range(1, n_parts)]
MODEL_VERSIONS = d_keys[:duration]


Ks = [1, 10, 20]
VM_K = Ks[2] # valid metric k, also used in heatmap matrix
VALID_METRIC = 'Recall@'+str(VM_K)
SEED = 2020
USE_GPU = True
SHOW_PROGRESS = False
SAVE_DATASET = False 

# dataset_save_path (str): The path of saved dataset. The tool will attempt to load the dataset from this path. If it equals to None, the tool will try to load the dataset from {checkpoint_dir}/{dataset}-{dataset_class_name}.pth.

# these are the default values
# TRAIN_NEG_SAMPLE_ARGS = {'distribution': 'uniform', 
#                          'sample_num': 1, 
#                          'alpha': 1.0, 
#                          'dynamic': False, 
#                          'candidate_num': 0}



SHUFFLE = False  # shuffle (bool): Whether or not to shuffle the training data before each epoch. Defaults to True.
EVAL_ARGS = {'split': {'LS': 'test_only'}, # leave-one-out sample type ['valid_and_test', 'valid_only', 'test_only']
                    'group_by': 'user',
                    'order': 'TO', # order (str): decides how we sort the data in .inter. random ordering or time ordering
                    'mode': 'uni100'}

METRICS = ['Recall', 'MRR', 'NDCG', 'Hit', 'Precision', 'GiniIndex', 'TailPercentage']

FILENAME_VERSION = '_GPU_ET_LS.t_UD_SF_TO_UM.100'

# BPR

In [ ]:
# model_name = 'BPR'

# for part in MODEL_VERSIONS:
#     print('\n\n'+part)
    
#     # current data (ith part of the dataset) to feed the pre-trained model,
#     # the result is referred to as "model part i" or "current model"
#     current_dataset_name = base_dataset_name+part
#     current_checkpoint_dir = save_path+current_dataset_name

#     parameter_dict = {  'dataset': current_dataset_name+'.inter',
#                         'use_gpu':USE_GPU,

#                         ## Environment settings https://recbole.io/docs/user_guide/config/environment_settings.html
#                         'seed':SEED,
#                         'state':'INFO' if SHOW_PROGRESS else 'ERROR', # ['INFO', 'DEBUG', 'WARNING', 'ERROR', 'CRITICAL']
#                         'data_path': save_path, # The path of input dataset.
#                         'save_dataset':SAVE_DATASET, 
#                         'checkpoint_dir':current_checkpoint_dir, # The path to save checkpoint file.
#                         'show_progress': SHOW_PROGRESS,
#                         'shuffle': SHUFFLE,

#                         ## Data settings https://recbole.io/docs/user_guide/config/data_settings.html
#                         'load_col': {'inter': ['user_id', 'item_id', 'timestamp']},
#                         # 'user_inter_num_interval':'[1,inf)',
#                         'benchmark_filename': BENCHMARK_FILENAMES,
                        
#                         ## Training settings https://recbole.io/docs/user_guide/config/training_settings.html
#                         # 'train_neg_sample_args': TRAIN_NEG_SAMPLE_ARGS,
                        
#                         ## Evaluation settings https://recbole.io/docs/user_guide/config/evaluation_settings.html
#                         'eval_args': EVAL_ARGS,
#                         'metrics': METRICS, 
#                         'topk':Ks,
#                         'valid_metric':VALID_METRIC          
#                     }


#     # current_config,current_logger,current_dataset,current_train_data, current_valid_data, current_test_data
#     current_config,\
#         current_logger, _,\
#             current_train_data,\
#                 current_valid_data,\
#                     current_test_data = setup_config_and_dataset(model_name,
#                                                                 current_dataset_name,
#                                                                 parameter_dict)

#     # model loading and initialization
#     current_model = BPR(current_config, current_train_data.dataset).to(current_config['device'])
#     current_logger.info(current_model)


#     # trainer loading and initialization
#     trainer = Trainer(current_config, current_model)
    
    
#     # model training
#     best_valid_score, best_valid_result = trainer.fit(current_train_data, current_valid_data)
#     print('\n\nTraining best results')
#     print('best_valid_score: ', best_valid_score)
#     print('best_valid_result: ', best_valid_result)


#     # main diagonal eval
#     test_result = evaluate(trainer, current_test_data)
#     save_evaluation_results(test_result, 
#                             current_config['checkpoint_dir'], 
#                             get_evaluation_results_filename(current_config['model'], 
#                                                             current_dataset_name, 
#                                                             part, 
#                                                             FILENAME_VERSION))


#     # evaluate in all testsets
#     test_full_data_sections = get_test_full_data_sections_with_names(model_version=part,
#                                             base_dataset_name=base_dataset_name,
#                                             models_versions=MODEL_VERSIONS)[:-1]
#     print(test_full_data_sections)

#     for testset_name in test_full_data_sections:

#         _,_,\
#             _,trainset,_,\
#                 testset = setup_config_and_dataset(model_name,
#                                                     testset_name,
#                                                     parameter_dict)

#         # When calculate ItemCoverage metrics, we need to run this code for set item_nums in eval_collector.
#         trainer.eval_collector.data_collect(trainset)
        
#         # model evaluation
#         test_result = evaluate(trainer, testset) # bc is the trainer that was just feed new data
#         save_evaluation_results(test_result, 
#                                 current_config['checkpoint_dir'],
#                                 get_evaluation_results_filename(current_config['model'], 
#                                                                 current_dataset_name, 
#                                                                 testset_name, 
#                                                                 FILENAME_VERSION))

In [ ]:
model_name = 'BPR'

recbole_train_each_eval_all(model_name=model_name,
                            model_versions=MODEL_VERSIONS,
                            save_path=save_path,
                            base_filename=base_filename,
                            specs_str=specs_str,
                            filename_version=FILENAME_VERSION,

                            use_gpu=USE_GPU,
                            seed=SEED,
                            show_progress=SHOW_PROGRESS,
                            save_dataset=SAVE_DATASET,
                            shuffle=SHUFFLE,
                            benchmark_filename=BENCHMARK_FILENAMES,
                            eval_args=EVAL_ARGS,
                            metrics=METRICS,
                            ks=Ks,
                            valid_metric=VALID_METRIC,
                        
                            part_shift_incl=False)

## recall heatmap

In [ ]:
model_name = 'BPR'

results_matrix = get_full_models_results_matrix(model_name=model_name,
                                    models_versions=MODEL_VERSIONS,
                                    base_dataset_name=base_dataset_name,
                                    save_path=save_path,
                                    metric='recall@'+str(VM_K),
                                    filename_version=FILENAME_VERSION)

recall_heatmap(results_matrix,
               round_point=4,
               title=model_name+' - Goodreads - 50% Random Drift - '+VALID_METRIC, 
               filepath='images/goodreads/',
               filename=base_dataset_name+FILENAME_VERSION+'_'+model_name+'_'+VALID_METRIC)

# NeuMF

In [ ]:
model_name = 'NeuMF'

recbole_train_each_eval_all(model_name=model_name,
                            model_versions=MODEL_VERSIONS,
                            save_path=save_path,
                            base_filename=base_filename,
                            specs_str=specs_str,
                            filename_version=FILENAME_VERSION,

                            use_gpu=USE_GPU,
                            seed=SEED,
                            show_progress=SHOW_PROGRESS,
                            save_dataset=SAVE_DATASET,
                            shuffle=SHUFFLE,
                            benchmark_filename=BENCHMARK_FILENAMES,
                            eval_args=EVAL_ARGS,
                            metrics=METRICS,
                            ks=Ks,
                            valid_metric=VALID_METRIC,
                        
                            part_shift_incl=False)

## recall heatmap

In [ ]:
model_name = 'NeuMF'

results_matrix = get_full_models_results_matrix(model_name=model_name,
                                    models_versions=MODEL_VERSIONS,
                                    base_dataset_name=base_dataset_name,
                                    save_path=save_path,
                                    metric='recall@'+str(VM_K),
                                    filename_version=FILENAME_VERSION)

recall_heatmap(results_matrix,
               round_point=4,
               title='NeuMF - Goodreads - 50% Random Drift - '+VALID_METRIC, 
               filepath='images/goodreads/',
               filename=base_dataset_name+FILENAME_VERSION+'_'+model_name+'_'+VALID_METRIC)

# Pop

In [ ]:
model_name = 'Pop'

recbole_train_each_eval_all(model_name=model_name,
                            model_versions=MODEL_VERSIONS,
                            save_path=save_path,
                            base_filename=base_filename,
                            specs_str=specs_str,
                            filename_version=FILENAME_VERSION,

                            use_gpu=USE_GPU,
                            seed=SEED,
                            show_progress=SHOW_PROGRESS,
                            save_dataset=SAVE_DATASET,
                            shuffle=SHUFFLE,
                            benchmark_filename=BENCHMARK_FILENAMES,
                            eval_args=EVAL_ARGS,
                            metrics=METRICS,
                            ks=Ks,
                            valid_metric=VALID_METRIC,
                        
                            part_shift_incl=False)

## recall heatmap

In [ ]:
model_name = 'Pop'

results_matrix = get_full_models_results_matrix(model_name=model_name,
                                    models_versions=MODEL_VERSIONS,
                                    base_dataset_name=base_dataset_name,
                                    save_path=save_path,
                                    metric='recall@'+str(VM_K),
                                    filename_version=FILENAME_VERSION)

recall_heatmap(results_matrix,
               round_point=4,
               title=model_name+' - Goodreads - 50% Random Drift - '+VALID_METRIC, 
               filepath='images/goodreads/',
               filename=base_dataset_name+FILENAME_VERSION+'_'+model_name+'_'+VALID_METRIC)

# ItemKNN

In [ ]:
model_name = 'ItemKNN'

recbole_train_each_eval_all(model_name=model_name,
                            model_versions=MODEL_VERSIONS,
                            save_path=save_path,
                            base_filename=base_filename,
                            specs_str=specs_str,
                            filename_version=FILENAME_VERSION,

                            use_gpu=USE_GPU,
                            seed=SEED,
                            show_progress=SHOW_PROGRESS,
                            save_dataset=SAVE_DATASET,
                            shuffle=SHUFFLE,
                            benchmark_filename=BENCHMARK_FILENAMES,
                            eval_args=EVAL_ARGS,
                            metrics=METRICS,
                            ks=Ks,
                            valid_metric=VALID_METRIC,
                        
                            part_shift_incl=False)

## recall heatmap

In [ ]:
model_name = 'Pop'

results_matrix = get_full_models_results_matrix(model_name=model_name,
                                    models_versions=MODEL_VERSIONS,
                                    base_dataset_name=base_dataset_name,
                                    save_path=save_path,
                                    metric='recall@'+str(VM_K),
                                    filename_version=FILENAME_VERSION)

recall_heatmap(results_matrix,
               round_point=4,
               title=model_name+' - Goodreads - 50% Random Drift - '+VALID_METRIC, 
               filepath='images/goodreads/',
               filename=base_dataset_name+FILENAME_VERSION+'_'+model_name+'_'+VALID_METRIC)